In [1]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
import optuna
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha import logger, WindowGenerator, LGBMLR_Runner
from vnpy.trader.constant import Interval
from datetime import datetime
from pathlib import Path
from vnpy.alpha.strategy import BacktestingEngine2
import vnpy.alpha.strategy.strategies.equity_demo_strategy2 as equity_demo_strategy2
EquityDemoStrategy = equity_demo_strategy2.EquityDemoStrategy2

C:\veighna_studio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================================
# Cell 2: 配置与初始化
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

lab = AlphaLab(str(LAB_PATH))
dataset = lab.load_dataset('v100')

In [3]:
# ============================================================================
# Cell 3: 生成 WalkForward 窗口
# ============================================================================
windows = WindowGenerator.generate(
    start=datetime(2018, 1, 1),
    end=datetime(2026, 5, 8),
    train_years=3.0,
    valid_years=1.0,
    test_years=1.0,
    step_years=1.0,
)

print(f"生成 {len(windows)} 个窗口:")
for w in windows:
    print(
        f"  Win{w.index}: "
        f"train={w.train_start.date()}~{w.train_end.date()}, "
        f"valid={w.valid_start.date()}~{w.valid_end.date()}, "
        f"test={w.test_start.date()}~{w.test_end.date()}"
    )

生成 5 个窗口:
  Win0: train=2018-01-01~2021-01-01, valid=2021-01-01~2022-01-01, test=2022-01-01~2023-01-01
  Win1: train=2019-01-01~2022-01-01, valid=2022-01-01~2023-01-01, test=2023-01-01~2024-01-01
  Win2: train=2020-01-01~2023-01-01, valid=2023-01-01~2024-01-01, test=2024-01-01~2025-01-01
  Win3: train=2021-01-01~2024-01-01, valid=2024-01-01~2025-01-01, test=2025-01-01~2026-01-01
  Win4: train=2022-01-01~2025-01-01, valid=2025-01-01~2026-01-01, test=2026-01-01~2026-05-08


In [4]:
def objective(trial):
    model_params = {
        "num_leaves": trial.suggest_int("num_leaves", 256, 2048, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 100, 1000),
        'max_depth': -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.0005, 0.01, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 100.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 100.0),

        }
    strategy_params={
        'top_k': 5,
        'min_days': 5,
        'cash_ratio': 0.95,
        'open_rate': 0.0005,
        'close_rate': 0.0015,
        'min_commission': 5,
        'slippage': 0.0006,
        'price_add': 0.05,
    }
    backtest_params={
        'interval': Interval.DAILY,
        'capital': 100_000,
        'risk_free': 0.0,
        'annual_days': 240,
        'min_commission': 5.0,
        'slippage': 0.0006,
        'adjust_type': 'none',
    }

    n_quantiles = trial.suggest_int("n_quantiles", 10, 60)

    runner = LGBMLR_Runner(
    name='lgb_baseline',
    lab=lab,
    dataset=dataset,
    engine_class=BacktestingEngine2,
    strategy_class=EquityDemoStrategy,
    windows = windows,
    model_params=model_params,
    strategy_params=strategy_params,
    backtest_params=backtest_params,
    benchmark_symbol='000300.SSE',
    n_quantiles=n_quantiles,
    seed=42,
)

    stats = runner.run(runner.windows, log = False)
    sharpe = stats['sharpe_ratio']
    return sharpe

In [5]:
study = optuna.create_study(
    direction='maximize',
    study_name='LGBMLR_optimization',
    storage=None,                            # 可改为 SQLite 路径持久化
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=1,
        n_warmup_steps=1,
        interval_steps=1
    ),
    sampler=optuna.samplers.TPESampler(seed=42),
)

# 执行优化（根据时间/算力调整 n_trials）
n_trials = 3
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

logger.info('\n===== Optuna 优化结果 =====')
logger.info(f'最佳 trial: {study.best_trial.number}')
logger.info(f'最佳夏普: {study.best_value}')
logger.info('最佳参数:')
for key, value in study.best_params.items():
    logger.info(f'  {key}: {value}')

[I 2026-05-29 00:03:18,379] A new study created in memory with name: LGBMLR_optimization
  0%|          | 0/3 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[106]	train's ndcg@5: 0.475926	train's ndcg@10: 0.459134	valid's ndcg@5: 0.369276	valid's ndcg@10: 0.364764
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[20]	train's ndcg@5: 0.455652	train's ndcg@10: 0.450803	valid's ndcg@5: 0.374561	valid's ndcg@10: 0.370751
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[5]	train's ndcg@5: 0.43838	train's ndcg@10: 0.428922	valid's ndcg@5: 0.342695	valid's ndcg@10: 0.344607
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[21]	train's ndcg@5: 0.439904	train's ndcg@10: 0.429942	valid's ndcg@5: 0.400936	valid's ndcg@10: 0.395908
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration 


Best trial: 0. Best value: -0.707597:  33%|███▎      | 1/3 [00:39<01:19, 39.77s/it]

[I 2026-05-29 00:03:58,144] Trial 0 finished with value: -0.7075967975460378 and parameters: {'num_leaves': 557, 'min_data_in_leaf': 956, 'learning_rate': 0.0044803926826840635, 'feature_fraction': 0.7993292420985183, 'bagging_fraction': 0.5780093202212182, 'bagging_freq': 2, 'lambda_l1': 5.8083612168199465, 'lambda_l2': 86.61761457749351, 'n_quantiles': 40}. Best is trial 0 with value: -0.7075967975460378.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[19]	train's ndcg@5: 0.460466	train's ndcg@10: 0.448586	valid's ndcg@5: 0.375116	valid's ndcg@10: 0.369007
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[45]	train's ndcg@5: 0.470098	train's ndcg@10: 0.45901	valid's ndcg@5: 0.381761	valid's ndcg@10: 0.37148
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[13]	train's ndcg@5: 0.446376	train's nd


Best trial: 1. Best value: -0.406956:  67%|██████▋   | 2/3 [01:41<00:52, 52.61s/it]

[I 2026-05-29 00:04:59,746] Trial 1 finished with value: -0.40695621714078234 and parameters: {'num_leaves': 1116, 'min_data_in_leaf': 118, 'learning_rate': 0.009138013915892867, 'feature_fraction': 0.9162213204002109, 'bagging_fraction': 0.6061695553391381, 'bagging_freq': 2, 'lambda_l1': 18.34045098534338, 'lambda_l2': 30.42422429595377, 'n_quantiles': 36}. Best is trial 1 with value: -0.40695621714078234.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[11]	train's ndcg@5: 0.434699	train's ndcg@10: 0.426728	valid's ndcg@5: 0.358731	valid's ndcg@10: 0.360847
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[15]	train's ndcg@5: 0.433958	train's ndcg@10: 0.43333	valid's ndcg@5: 0.361681	valid's ndcg@10: 0.363301
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3]	train's ndcg@5: 0.406258	train's n


Best trial: 1. Best value: -0.406956: 100%|██████████| 3/3 [02:23<00:00, 47.85s/it]


[I 2026-05-29 00:05:41,938] Trial 2 finished with value: -0.7461463371432145 and parameters: {'num_leaves': 628, 'min_data_in_leaf': 362, 'learning_rate': 0.003126143958203107, 'feature_fraction': 0.569746930326021, 'bagging_fraction': 0.6460723242676091, 'bagging_freq': 4, 'lambda_l1': 45.606998421703594, 'lambda_l2': 78.51759613930136, 'n_quantiles': 20}. Best is trial 1 with value: -0.40695621714078234.


TypeError: 'Logger' object is not callable

In [9]:
logger.info('\n使用最佳参数训练模型...')
best_params = study.best_params
model_params = {
        "num_leaves": best_params['num_leaves'],
        "min_data_in_leaf": best_params['min_data_in_leaf'],
        'max_depth': -1,
        "learning_rate": best_params['learning_rate'],
        "feature_fraction": best_params['feature_fraction'],
        "bagging_fraction": best_params['bagging_fraction'],
        "bagging_freq": best_params['bagging_freq'],
        "lambda_l1": best_params['lambda_l1'],
        "lambda_l2": best_params['lambda_l2'],

        }
strategy_params={
    'top_k': 5,
    'min_days': 5,
    'cash_ratio': 0.95,
    'open_rate': 0.0005,
    'close_rate': 0.0015,
    'min_commission': 5,
    'slippage': 0.0006,
    'price_add': 0.05,
}
backtest_params={
    'interval': Interval.DAILY,
    'capital': 100_000,
    'risk_free': 0.0,
    'annual_days': 240,
    'min_commission': 5.0,
    'slippage': 0.0006,
    'adjust_type': 'none',
}

n_quantiles = best_params['n_quantiles']

runner = LGBMLR_Runner(
name='lgb_baseline',
lab=lab,
dataset=dataset,
engine_class=BacktestingEngine2,
strategy_class=EquityDemoStrategy,
windows = windows,
model_params=model_params,
strategy_params=strategy_params,
backtest_params=backtest_params,
benchmark_symbol='000300.SSE',
n_quantiles=n_quantiles,
seed=42,
)

stats = runner.run(runner.windows, log = True)

2026-05-29 00:15:36.706 | INFO     | __main__:<module>:1 - 
使用最佳参数训练模型...
2026-05-29 00:15:36.708 | INFO     | vnpy.alpha.walkforward.runner:run_window:404 - [Window 0] 执行：
2026-05-29 00:15:36.709 | INFO     | vnpy.alpha.walkforward.runner:extract_data:298 - 提取 LambdaRank 训练数据...
2026-05-29 00:15:36.751 | INFO     | vnpy.alpha.dataset.template:extract_lambdarank_data:457 - TRAIN, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35]
2026-05-29 00:15:36.771 | INFO     | vnpy.alpha.dataset.template:extract_lambdarank_data:477 - TRAIN, X.shape=(219000, 65)
2026-05-29 00:15:36.773 | INFO     | vnpy.alpha.walkforward.runner:extract_data:303 - 提取验证数据...
2026-05-29 00:15:36.797 | INFO     | vnpy.alpha.dataset.template:extract_lambdarank_data:457 - VALID, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35]
2026-05-29 00:15:36.805 | INFO     | vnpy.alpha.dataset.template

Training until validation scores don't improve for 100 rounds


2026-05-29 00:15:42.898 | INFO     | vnpy.alpha.walkforward.runner:train_model:377 - 训练完成！最佳迭代轮数: 19
2026-05-29 00:15:42.900 | INFO     | vnpy.alpha.walkforward.runner:train_model:379 - VALID BEST NDCG: OrderedDict({'ndcg@5': np.float64(0.3751164676421404), 'ndcg@10': np.float64(0.36900656870362286)})
2026-05-29 00:15:42.900 | INFO     | vnpy.alpha.walkforward.runner:train_model:380 - TRAIN NDCG: OrderedDict({'ndcg@5': np.float64(0.4604656182955852), 'ndcg@10': np.float64(0.44858627405935453)})
2026-05-29 00:15:42.902 | INFO     | vnpy.alpha.walkforward.runner:predict_signal:386 - ---在测试集上预测---
2026-05-29 00:15:42.924 | INFO     | vnpy.alpha.walkforward.runner:predict_signal:390 - 预测完成，预测样本数:72600
2026-05-29 00:15:42.925 | INFO     | vnpy.alpha.walkforward.runner:predict_signal:396 - siganl.shape: (72600, 3)
2026-05-29 00:15:42.926 | INFO     | vnpy.alpha.walkforward.runner:predict_signal:397 - siganl:
2026-05-29 00:15:42.926 | INFO     | vnpy.alpha.walkforward.runner:predict_signal:39

Early stopping, best iteration is:
[19]	train's ndcg@5: 0.460466	train's ndcg@10: 0.448586	valid's ndcg@5: 0.375116	valid's ndcg@10: 0.369007
Evaluated only: ndcg@5


2026-05-29 00:15:43.056 | INFO     | vnpy.alpha.dataset.template:extract_lambdarank_data:457 - TEST, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35]
2026-05-29 00:15:43.064 | INFO     | vnpy.alpha.dataset.template:extract_lambdarank_data:477 - TEST, X.shape=(72600, 65)
2026-05-29 00:15:43.066 | INFO     | vnpy.alpha.walkforward.runner:train_model:332 - 开始训练 LambdaRank 模型...


Training until validation scores don't improve for 100 rounds


KeyboardInterrupt: 